# Interactive Vertebra Classification

This notebook provides an interactive way to load a vertebra mesh, embed it into the latent space using the pretrained DeepSDF model, and classify it across multiple taxonomic tiers (Species, Genus, Family) and spinal positions (Cervical, Thoracic, Lumbar).

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import pyvista as pv
from pyvistaqt import BackgroundPlotter

from NSM.datasets import SDFSamples
from NSM.mesh import create_mesh
from NSM.helper_funcs import load_config, load_model_and_latents, extract_species_prefix, convert_ply_to_vtk
from classify_vertebrae import optimize_latent
from taxonomy_utils import parse_taxonomy_from_filename
from metric_learning import LatentMetricLearner
from supervised_classifiers import train_classifiers, predict_classifiers
from hierarchical_classification import HierarchicalMultiTaskClassifier
from NSM.optimization import find_similar_cos, pca_initialize_latent
import warnings
warnings.filterwarnings('ignore')

## 1. Loader & Configuration

In [ ]:
# Choose model checkpoint
TRAIN_DIR = "run_v56"
CKPT = "2000"

device = torch.device('cuda:0' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

os.chdir(TRAIN_DIR)
config = load_config('model_params_config.json')
config['device'] = device
model, latent_ckpt, latent_codes = load_model_and_latents(f"model/{CKPT}.pth", f"latent_codes/{CKPT}.pth", config, device)

mean_latent = latent_codes.mean(dim=0, keepdim=True)
train_paths = config['list_mesh_paths']
all_vtk_files = [os.path.basename(f) for f in train_paths]
print(f"Loaded {len(latent_codes)} reference latents.")

## 2. Prepare Reference Dataset

In [ ]:
X_train_raw = latent_codes.cpu().numpy()
y_labels = []

for f in all_vtk_files:
    parsed = parse_taxonomy_from_filename(f)
    if parsed:
        y_labels.append({
            'species': parsed['species'],
            'genus': parsed['genus'],
            'family': parsed['family'],
            'position': parsed['position']
        })
    else:
        y_labels.append({'species': 'unknown', 'genus': 'unknown', 'family': 'unknown', 'position': 'unknown'})

# Define Target: For multi-task, we can combine species + position
valid_idx = [i for i, lbl in enumerate(y_labels) if lbl['species'] != 'unknown' and lbl['position'] is not None]
X_train = X_train_raw[valid_idx]
y_train_species = np.array([y_labels[i]['species'] for i in valid_idx])
y_train_position = np.array([y_labels[i]['position'] for i in valid_idx])
y_train_multi = np.column_stack((y_train_species, y_train_position))

print(f"Prepared {len(X_train)} labeled samples for building metrics.")

### Fit Supervised Extractor (NCA) & Classifier
We use Metric Learning to enforce semantic separation before training the hierarchal model.

In [ ]:
metric_learner = LatentMetricLearner(method='NCA', max_iter=50)
print("Fitting Metric Learner (this may take a moment)...")
X_train_transformed = metric_learner.fit_transform(X_train, y_train_species)

print("Training Hierarchical Multi-Task Classifier...")
multi_clf = HierarchicalMultiTaskClassifier()
multi_clf.fit(X_train_transformed, y_train_multi)
print("Ready!")

## 3. Interactive Widget

In [ ]:
import glob

mesh_options = glob.glob("../vertebrae_meshes/*.vtk")
mesh_names = [os.path.basename(m) for m in mesh_options]

dropdown_mesh = widgets.Dropdown(
    options=dict(zip(mesh_names, mesh_options)),
    description='Test Mesh:',
    disabled=False,
)

button_run = widgets.Button(description="Evaluate Mesh")
output_log = widgets.Output()

def on_evaluate_clicked(b):
    with output_log:
        clear_output()
        mesh_path = dropdown_mesh.value
        fname = os.path.basename(mesh_path)
        print(f"Evaluating {fname}...")
        
        if '.ply' in mesh_path:
            _, mesh_path = convert_ply_to_vtk(mesh_path)
            
        sdf_dataset = SDFSamples(
            list_mesh_paths=[mesh_path],
            multiprocessing=False,
            subsample=config["samples_per_object_per_batch"],
            n_pts=config["n_pts_per_object"],
            p_near_surface=config['percent_near_surface'],
            p_further_from_surface=config['percent_further_from_surface'],
            sigma_near=config['sigma_near'],
            sigma_far=config['sigma_far'],
            rand_function=config['random_function'], 
            center_pts=config['center_pts'],
            norm_pts=config['normalize_pts'],
            fix_mesh=config['fix_mesh']
        )
        
        sample_dict, _ = sdf_dataset[0]
        points = sample_dict['xyz'].to(device)
        sdf_vals = sample_dict['gt_sdf']
        
        print("Optimizing latents... (Wait ~30s)")
        # Hack optimize_latent to local scope
        init_latent_torch = mean_latent
        latent = init_latent_torch.clone().detach().requires_grad_()
        optimizer = torch.optim.Adam([latent], lr=1e-3)
        sdf_vals = sdf_vals.to(device)
        model.to(device)
        for i in range(1000):
            optimizer.zero_grad()
            # Assuming monkey patch fast path is available
            pred_sdf = model(latent=latent.squeeze(), xyz=points)
            loss = torch.nn.functional.l1_loss(pred_sdf.squeeze(), sdf_vals)
            loss.backward()
            optimizer.step()
            
        latent_novel = latent.detach()
        print("Done optimizing.")
        
        novel_vec_raw = latent_novel.cpu().numpy()
        novel_vec_t = metric_learner.transform(novel_vec_raw)
        
        preds = multi_clf.predict(novel_vec_t)[0]
        probs = multi_clf.predict_proba(novel_vec_t)
        
        print("\n=== Multi-Task Predictions ===")
        print(f"Predicted Species : {preds[0]} (Confidence: {max(probs[0][0]):.2%})")
        print(f"Predicted Position: {preds[1]} (Confidence: {max(probs[1][0]):.2%})")
        
        # Top-5 similar by cosine
        similar_ids_cos, dists = find_similar_cos(latent_novel, latent_codes, top_k=5, n_std=5, device=device)
        print("\n=== Nearest Neighbors (Cosine in Orig Latent Space) ===")
        for rank, (idx, d) in enumerate(zip(similar_ids_cos, dists)):
            print(f"{rank+1}. {all_vtk_files[idx]} (Dist: {d:.4f})")
            
        print("\n(Interactive plotting disabled in this inline log, run cells below for PyVista visualization)")

button_run.on_click(on_evaluate_clicked)
display(dropdown_mesh, button_run, output_log)